
#Static Equilibrium using SMT solvers

Working out how hard a support is pushing on a loaded beam normally means choosing a
clever point to take torques about, so that the unknowns you do not care about drop out
of the equation. In this notebook we will look at SMT solvers and see how they let us
skip that choice entirely. We will write down the force balance and the torque balance
as they stand, keeping every term, and let Z3 find the unknown forces. We will also
build a beam that cannot be held at all, and see how the solver reports it.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver, cross, draw_beam
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Reals

Let's use Z3 to solve problems involving real numbers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Real('x') # declairing that x is a real number named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

## Forces and Torques on a Beam

A structure that is not moving is said to be in **static equilibrium**, and that puts
two conditions on the forces acting on it. The first is familiar: the forces have to
cancel, or the structure would accelerate away.

$$ \sum \vec{F} = 0 $$

The second is about rotation. Even when the forces cancel, a structure can still spin,
so the **torques** have to cancel as well.

$$ \sum \vec{\tau} = 0 $$

Between them, these two conditions are usually enough to work out the forces we cannot
measure directly, such as how hard a support is pushing back. We will write both of
them as constraints and let Z3 find those unknown forces.

### Torque

The torque a force produces depends on where it is applied. A force $\vec{F}$ applied
at a position $\vec{r}$, measured from whatever point we are taking torques about,
produces

$$ \vec{\tau} = \vec{r} \times \vec{F} $$

Everything in this notebook lies in a plane, and for two vectors in a plane the cross
product has only one component:

$$ \tau = r_x F_y - r_y F_x $$

We have defined a function that computes this, so we can write torques without
expanding them by hand. It takes each vector as a two-element list, `[x, y]`.

In [ ]:
r = [2, 0]      # 2 metres to the right of the point we are taking torques about
F = [0, -200]   # a 200 N force pulling straight down

print(cross(r, F), "N*m")

The result is negative, which tells us the direction: this force would rotate the
beam clockwise about that point. A force pushing up at the same place would give
$+400$ and rotate it the other way.

### A case we can check by eye

Before anything harder, let's set up a beam whose answer we already know.

A beam 4 metres long rests on a support at each end. It weighs 200 N, and all of that
weight acts at its midpoint, 2 metres along. By symmetry each support must be carrying
half of it, so we expect 100 N at each end.

We describe each force and each position as a pair of Z3 variables, one for the $x$
component and one for the $y$ component. The support at A is a pin, which can push in
any direction; the support at B is a roller, which can only push straight up.

In [ ]:
s = Solver()

# forces, as [x, y] pairs
FA = list(Reals('FAx FAy')) # the reaction at support A
FW = list(Reals('FWx FWy')) # the weight of the beam
FB = list(Reals('FBx FBy')) # the reaction at support B

# where each of those forces acts
rA = list(Reals('rAx rAy'))
rW = list(Reals('rWx rWy'))
rB = list(Reals('rBx rBy'))

# everything acts on the beam itself, which lies flat along y = 0
s.add(rA[1] == 0, rW[1] == 0, rB[1] == 0)
s.add(rA[0] == 0)
s.add(rW[0] == 2)
s.add(rB[0] == 4)

# the weight pulls straight down
s.add(FW[0] == 0)
s.add(FW[1] == -200)

# the roller at B can only push perpendicular to the beam
s.add(FB[0] == 0)

# the forces cancel
s.add(FA[0] + FW[0] + FB[0] == 0)
s.add(FA[1] + FW[1] + FB[1] == 0)

# the torques cancel, taken about the origin
s.add(cross(rA, FA) + cross(rW, FW) + cross(rB, FB) == 0)

showSolver(s)
print(s.check())

In [ ]:
solution = s.model()

print(f"support A pushes up with {solution[FA[1]]} N")
print(f"support B pushes up with {solution[FB[1]]} N")

100 N at each end, as expected. Notice that we never picked a point to take torques
about and then argued that one of the unknowns drops out, which is the usual first move
when doing this by hand. We took torques about the origin, where one of the unknowns
happens to sit, and kept every term.

In [ ]:
draw_beam(solution, [rA, rW, rB], [FA, FW, FB], ["F_A", "W", "F_B"])

### A beam that cannot be held

Now take support B away and move the weight off-centre, 3 metres along, so that the
beam is held only by the pin at A.

A pin can push in any direction, so the forces can still be made to cancel. The
question is whether the torques can.

In [ ]:
s = Solver()

FA = list(Reals('FAx FAy'))
FW = list(Reals('FWx FWy'))

rA = list(Reals('rAx rAy'))
rW = list(Reals('rWx rWy'))

s.add(rA[1] == 0, rW[1] == 0)
s.add(rA[0] == 0)
s.add(rW[0] == 3)

s.add(FW[0] == 0)
s.add(FW[1] == -200)

# the forces cancel
s.add(FA[0] + FW[0] == 0)
s.add(FA[1] + FW[1] == 0)

# the torques cancel
s.add(cross(rA, FA) + cross(rW, FW) == 0)

showSolver(s)
print(s.check())

"unsat" means no assignment of forces satisfies all of the constraints at once. The
force balance alone was satisfiable a moment ago, so the torque condition is what
cannot be met: the pin sits at the origin and therefore contributes no torque at all,
while the weight contributes $-600$ N·m, and nothing is left to cancel it. The beam
would rotate about the pin.

This is the kind of answer that is easy to miss when working by hand, because the force
balance looks perfectly healthy on its own. Here the solver refuses the whole system
rather than handing back an answer that only satisfies half of the physics.

### Your turn

Here is the beam we actually want to solve.

It is 4 metres long with a support at each end, again a pin at A and a roller at B. It
weighs 200 N acting at its midpoint, and a 500 N load sits on it 3 metres along, closer
to B than to A.

Everything is written below except the torque condition. **Replace the marked line**
with the constraint saying that the torques of all four forces, taken about the origin,
cancel. The function `cross` and the position and force pairs are all defined above
it.

In [ ]:
s = Solver()

# forces, as [x, y] pairs
FA = list(Reals('FAx FAy')) # the reaction at support A
FW = list(Reals('FWx FWy')) # the weight of the beam
FP = list(Reals('FPx FPy')) # the load sitting on the beam
FB = list(Reals('FBx FBy')) # the reaction at support B

# where each of those forces acts
rA = list(Reals('rAx rAy'))
rW = list(Reals('rWx rWy'))
rP = list(Reals('rPx rPy'))
rB = list(Reals('rBx rBy'))

# everything acts on the beam, which lies flat along y = 0
s.add(rA[1] == 0, rW[1] == 0, rP[1] == 0, rB[1] == 0)
s.add(rA[0] == 0)
s.add(rW[0] == 2)
s.add(rP[0] == 3)
s.add(rB[0] == 4)

# the weight and the load both pull straight down
s.add(FW[0] == 0)
s.add(FW[1] == -200)
s.add(FP[0] == 0)
s.add(FP[1] == -500)

# the roller at B can only push perpendicular to the beam
s.add(FB[0] == 0)

# the forces cancel
s.add(FA[0] + FW[0] + FP[0] + FB[0] == 0)
s.add(FA[1] + FW[1] + FP[1] + FB[1] == 0)

# the torques cancel, taken about the origin
s.add(True) # REPLACE THIS LINE

showSolver(s)
print(s.check())

In [ ]:
solution = s.model()

print(f"support A pushes up with {solution[FA[1]]} N")
print(f"support B pushes up with {solution[FB[1]]} N")

In [ ]:
draw_beam(solution, [rA, rW, rP, rB], [FA, FW, FP, FB], ["F_A", "W", "P", "F_B"])

The two reactions come to 225 N and 475 N, which add up to the 700 N pulling down, as
the force balance requires. They are not equal, and the larger one is at B, because the
500 N load sits closer to that end and so more of it is carried there.


###Congratulations! You just used an SMT solver to find the forces holding up a beam!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm